In [ ]:
!pip install pycolmap
!pip install git+https://github.com/cvg/LightGlue.git

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import os
import shutil
import numpy as np
from scipy.spatial.transform import Rotation as R
import cv2
import torch
import pycolmap
import array
from lightglue import ALIKED  # Настройте в зависимости от фактического импорта
from lightglue import LightGlue  # Настройте в зависимости от фактического импорта

In [ ]:
data_path = "/kaggle/input/image-matching-challenge-2025"
test_path = os.path.join(data_path, "test")

# Загрузка sample_submission.csv
submission = pd.read_csv(os.path.join(data_path, "sample_submission.csv"))

In [ ]:
# Функция для разделения изображений на группы по размерам (для ETs)
def group_images_by_size(dataset, scene, group):
    image_sizes = {}
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            img = cv2.imread(image_path)
            if img is not None:
                size = img.shape[:2]
                size_key = f"{size[0]}x{size[1]}"
                if size_key not in image_sizes:
                    image_sizes[size_key] = []
                image_sizes[size_key].append((idx, row))
    return image_sizes

In [ ]:
def process_group(dataset, scene, group, indices):
    project_dir = f"/kaggle/working/project_{dataset}_{scene}_{indices[0]}"
    os.makedirs(project_dir, exist_ok=True)
    db_path = os.path.join(project_dir, "database.db")
    
    # Инициализация базы данных
    db = pycolmap.Database(db_path)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    aliked = ALIKED().to(device)
    matcher = LightGlue(features='aliked').to(device)
    
    image_ids = {}
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            shutil.copy(image_path, project_dir)
            img = cv2.imread(image_path)
            h, w = img.shape[:2]
            focal = 1.2 * max(w, h)
            
            # Создание камеры
            camera = pycolmap.Camera(
                model="SIMPLE_PINHOLE",
                width=w,
                height=h,
                params=[focal, w/2, h/2]
            )
            camera_id = db.write_camera(camera)
            
            # Добавление изображения
            image = pycolmap.Image(
                name=row['image'],
                camera_id=camera_id,
                tvec=[0, 0, 0],
                qvec=[1, 0, 0, 0]
            )
            image_id = db.add_image(image)
            image_ids[row['image']] = image_id
    
    # Извлечение признаков
    for image_name, image_id in image_ids.items():
        img = cv2.imread(os.path.join(project_dir, image_name))
        img_tensor = torch.from_numpy(img).permute(2,0,1).float().to(device) / 255.0
        feats = aliked.extract(img_tensor)
        
        keypoints = feats['keypoints'].cpu().numpy()
        descriptors = feats['descriptors'].cpu().numpy()
        
        # Сохранение ключевых точек
        kp_array = np.hstack([keypoints, np.ones((len(keypoints), 1)), np.zeros((len(keypoints), 1))])
        db.add_keypoints(image_id, kp_array.astype(np.float64))
        
        # Сохранение дескрипторов
        db.add_descriptors(image_id, descriptors.astype(np.float32))
    
    # Сопоставление признаков
    image_list = list(image_ids.items())
    for i in range(len(image_list)):
        for j in range(i+1, len(image_list)):
            id1, (name1, image_id1) = i, image_list[i]
            id2, (name2, image_id2) = j, image_list[j]
            
            desc1 = db.get_descriptors(image_id1)
            desc2 = db.get_descriptors(image_id2)
            
            matches = matcher.match(
                torch.from_numpy(desc1).to(device),
                torch.from_numpy(desc2).to(device)
            )
            matches = matches['matches'].cpu().numpy().astype(np.uint32)
            db.add_matches(image_id1, image_id2, matches)
    
    # Реконструкция
    options = pycolmap.IncrementalPipelineOptions()
    options.min_num_matches = 15
    reconstructions = pycolmap.incremental_mapping(
        database_path=db_path,
        image_path=project_dir,
        output_path=os.path.join(project_dir, "sparse"),
        options=options
    )
    
    if reconstructions:
        best_rec = max(reconstructions.values(), key=lambda r: len(r.points3D))
        for image_name, image_id in image_ids.items():
            if image_id in best_rec.images:
                image = best_rec.images[image_id]
                rot = image.qvec2rotmat().flatten()
                tvec = image.tvec
                idx = group[group['image'] == image_name].index[0]
                submission.at[idx, 'rotation_matrix'] = ';'.join(map(str, rot))
                submission.at[idx, 'translation_vector'] = ';'.join(map(str, tvec))
    
    shutil.rmtree(project_dir)

In [ ]:
groups = submission.groupby(['dataset', 'scene'])

# Установка headless-режима для COLMAP (оставлено для совместимости, хотя не используется)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

In [ ]:
for (dataset, scene), group in groups:
    dataset_path = os.path.join(test_path, dataset)
    if not os.path.exists(dataset_path):
        print(f"Skipping {dataset}")
        continue
    
    if dataset == 'ETs':
        image_groups = group_images_by_size(dataset, scene, group)
        for size_key, size_group in image_groups.items():
            print(f"Processing {size_key}")
            temp_group = pd.DataFrame([row for _, row in size_group])
            temp_indices = [idx for idx, _ in size_group]
            process_group(dataset, scene, temp_group, temp_indices)
    else:
        process_group(dataset, scene, group, group.index)

submission.to_csv('submission.csv', index=False)